In [106]:
import numpy as np
import pandas as pd 

In [107]:
data_type = {"gameId": str, "teamId": str}
df_stat = pd.read_csv('../dataset/merged_team_stats.csv', dtype=data_type )
nb_games = "10"
df_stat

,teamId,gameId,GAME_DATE,DUEL,WL,NB_VICTOIRE,NB_DEFAITE,PCT_VICTOIRE,DUREE_MATCH,TIR REUSSI,...,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,1610612748,0021300002,2013-10-29,MIA vs. CHI,W,1,0,1.000,240,37,...,20.2,0.590,0.631,1.0,0.197,100.44,98.5,82.08,99.0,0.599
1,1610612747,0021300003,2013-10-29,LAL vs. LAC,W,1,0,1.000,240,42,...,19.2,0.527,0.551,1.0,0.195,102.72,98.5,82.08,99.0,0.503
2,1610612746,0021300003,2013-10-29,LAC @ LAL,L,0,1,0.000,240,41,...,16.3,0.542,0.553,1.0,0.200,102.72,98.5,82.08,98.0,0.497
3,1610612754,0021300001,2013-10-29,IND vs. ORL,W,1,0,1.000,240,34,...,22.3,0.528,0.570,1.0,0.198,99.74,94.0,78.33,94.0,0.661
4,1610612741,0021300002,2013-10-29,CHI @ MIA,L,0,1,0.000,240,35,...,19.4,0.464,0.510,1.0,0.198,100.44,98.5,82.08,98.0,0.401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24077,1610612739,0022301187,2024-04-14,CLE vs. CHA,L,48,34,0.585,240,44,...,14.4,0.559,0.567,1.0,0.196,97.40,97.0,80.83,97.0,0.427
24078,1610612761,0022301189,2024-04-14,TOR @ MIA,L,25,57,0.305,240,38,...,19.2,0.472,0.519,1.0,0.198,103.30,100.0,83.33,99.0,0.369
24079,1610612742,0022301196,2024-04-14,DAL @ OKC,L,50,32,0.610,240,32,...,14.3,0.371,0.412,1.0,0.199,107.76,105.5,87.92,105.0,0.246
24080,1610612754,0022301188,2024-04-14,IND vs. ATL,W,47,35,0.573,240,65,...,15.0,0.745,0.752,1.0,0.193,108.64,107.0,89.17,107.0,0.649


In [108]:
df_game_info = df_stat[["TOTAL_POINTS", "DUEL",'GAME_DATE']].copy()
df_game_info.loc[:,'teamName'] = df_stat['teamCity'] + " " + df_stat['teamName']
df_game_info

,TOTAL_POINTS,DUEL,GAME_DATE,teamName
0,107,MIA vs. CHI,2013-10-29,Miami Heat
1,116,LAL vs. LAC,2013-10-29,Los Angeles Lakers
2,103,LAC @ LAL,2013-10-29,Los Angeles Clippers
3,97,IND vs. ORL,2013-10-29,Indiana Pacers
4,95,CHI @ MIA,2013-10-29,Chicago Bulls
...,...,...,...,...
24077,110,CLE vs. CHA,2024-04-14,Cleveland Cavaliers
24078,103,TOR @ MIA,2024-04-14,Toronto Raptors
24079,86,DAL @ OKC,2024-04-14,Dallas Mavericks
24080,157,IND vs. ATL,2024-04-14,Indiana Pacers


In [109]:
def get_w_diff(team_stats, index):
    if index == 0 or team_stats['NB_VICTOIRE'][index-1] + team_stats['NB_DEFAITE'][index-1] == 82:
        return 0
    else:
        return team_stats['NB_VICTOIRE'][index-1] - team_stats['NB_DEFAITE'][index-1]
    
df_stat_last_perf = pd.DataFrame(columns=['gameId', 'teamId', 'GAME_DATE', 'DUEL', 'WL', 'DIFF_W' ,'NB_WIN_L'+nb_games, 'PCT_TIR_REUSSI_L'+nb_games,'PCT_3PTS_L'+nb_games, 
                           'PCT_LANCER_FRANC_L'+nb_games,'estimatedOffensiveRating_L'+nb_games, 'offensiveRating_L'+nb_games,
                           'estimatedDefensiveRating_L'+nb_games,
                           'defensiveRating_L'+nb_games,'estimatedNetRating_L'+nb_games, 'netRating_L'+nb_games, 'assistPercentage_L'+nb_games,
                           'assistToTurnover_L'+nb_games, 'assistRatio_L'+nb_games,'estimatedTeamTurnoverPercentage_L'+nb_games, 'turnoverRatio_L'+nb_games,
                           'effectiveFieldGoalPercentage_L'+nb_games,'trueShootingPercentage_L'+nb_games, 'estimatedPace_L'+nb_games, 'pace_L'+nb_games,
                           'pacePer40_L'+nb_games, 'PIE_L'+nb_games])

for team in list(set(df_stat['teamId'].tolist())):
    team_stats = df_stat[df_stat['teamId'] == team]
    team_stats.reset_index(drop=True, inplace=True)
    index = 0
    while index < len(team_stats):
        df_last_10_games = team_stats.iloc[max(0, index - int(nb_games)):index]
        df_last_10_games.reset_index(drop=True, inplace=True)
        new_row = {"gameId" :  team_stats['gameId'][index],
                   "teamId" : team_stats['teamId'][index],
                   "GAME_DATE" : team_stats['GAME_DATE'][index],
                   "DUEL" : team_stats['DUEL'][index],
                   "WL" : team_stats['WL'][index],
                   #"DIFF_W" : get_w_diff(team_stats, index),
                   "NB_WIN_L"+nb_games : (df_last_10_games['WL'] == 'W').sum() / int(nb_games),
                   "PCT_TIR_REUSSI_L"+nb_games : df_last_10_games['PCT_TIR_REUSSI'].mean(),
                   "PCT_3PTS_L"+nb_games : df_last_10_games['PCT_3PTS'].mean(),
                   "PCT_LANCER_FRANC_L"+nb_games : df_last_10_games['PCT_LANCER_FRANC'].mean(),
                   "estimatedOffensiveRating_L"+nb_games : df_last_10_games['estimatedOffensiveRating'].mean(),
                   "offensiveRating_L"+nb_games : df_last_10_games['offensiveRating'].mean(),
                   "estimatedDefensiveRating_L"+nb_games : df_last_10_games['estimatedDefensiveRating'].mean(),
                   "defensiveRating_L"+nb_games : df_last_10_games['defensiveRating'].mean(),
                   "estimatedNetRating_L"+nb_games : df_last_10_games['estimatedNetRating'].mean(),
                   "netRating_L"+nb_games : df_last_10_games['netRating'].mean(),
                   "assistPercentage_L"+nb_games : df_last_10_games['assistPercentage'].mean(),
                   "assistToTurnover_L"+nb_games : df_last_10_games['assistToTurnover'].mean(),
                   "assistRatio_L"+nb_games : df_last_10_games['assistRatio'].mean(),
                   "estimatedTeamTurnoverPercentage_L"+nb_games : df_last_10_games['estimatedTeamTurnoverPercentage'].mean(),
                   "turnoverRatio_L"+nb_games : df_last_10_games['turnoverRatio'].mean(),
                   "effectiveFieldGoalPercentage_L"+nb_games : df_last_10_games['effectiveFieldGoalPercentage'].mean(),
                   "trueShootingPercentage_L"+nb_games : df_last_10_games['trueShootingPercentage'].mean(),
                   "estimatedPace_L"+nb_games : df_last_10_games['estimatedPace'].mean(),
                   "pace_L"+nb_games : df_last_10_games['pace'].mean(),
                   "pacePer40_L"+nb_games : df_last_10_games['pacePer40'].mean(),
                   "PIE_L"+nb_games : df_last_10_games['PIE'].mean()}
        if index == 0 or team_stats['NB_VICTOIRE'][index-1] + team_stats['NB_DEFAITE'][index-1] == 82:
            new_row['DIFF_W'] = 0
        else :
            new_row['DIFF_W'] = team_stats['NB_VICTOIRE'][index-1] - team_stats['NB_DEFAITE'][index-1]
        df_stat_last_perf.loc[len(df_stat_last_perf)] = new_row
        index += 1
df_stat_last_perf

KeyboardInterrupt: 

In [ ]:
#df_stat_last_perf['POINTS'] = df_game_info
df_stat_last_perf = pd.merge(df_stat_last_perf, df_game_info, on=['DUEL', 'GAME_DATE'])
df_stat_last_perf

In [ ]:
df_stat_last_perf['W_CONFR_D'] = np.nan
df_stat_last_perf['GAME_DATE'] = pd.to_datetime(df_stat_last_perf['GAME_DATE'])
for team in list(set(df_stat['teamId'].tolist())):
    game_list_one_year_before = df_stat_last_perf[df_stat_last_perf['teamId'] == team].reset_index()
    for index, game in game_list_one_year_before.iterrows():
        adv = game['DUEL'].split()[2]
        game_list_against_adv = game_list_one_year_before[(game_list_one_year_before['GAME_DATE'] < game['GAME_DATE']) & (game_list_one_year_before['GAME_DATE'] > (game['GAME_DATE'] - pd.DateOffset(years=2))) & (game_list_one_year_before['DUEL'].str.contains(adv))]
        df_stat_last_perf.loc[game['index'], 'W_CONFR_D'] = (game_list_against_adv['WL'] == "W").sum()

In [ ]:
df_stat_last_perf

In [ ]:
df_stat_last_perf = df_stat_last_perf.sort_values(by='gameId')
df_stat_last_perf.reset_index(drop=True, inplace=True)
df_stat_last_perf

In [ ]:
for game_id in list(set(df_stat_last_perf['gameId'].tolist())):
    df_game = df_stat_last_perf[df_stat_last_perf['gameId'] == game_id]
    df_game.reset_index(inplace=True)
    if '@' in df_game['DUEL'].values[0]:
        df_stat_last_perf.loc[df_game['index'].values[0]] = df_game.loc[1]
        df_stat_last_perf.loc[df_game['index'].values[1]] = df_game.loc[0]
        
df_stat_last_perf

In [ ]:
df_stat_last_perf.to_csv('../dataset/team_stat_last_'+ nb_games +'.csv', index=False)